In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re

# **Read data**

In [ ]:
df = pd.read_csv('IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


# **Describe data**

In [ ]:
df.describe().T

,count,unique,top,freq
review,50000,49582,Loved today's show!!! It was a variety and not...,5
sentiment,50000,2,positive,25000


In [ ]:
df['sentiment'].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [ ]:
df.replace({"sentiment": {"positive": 1, "negative": 0}}, inplace=True)

# **Preprocess data**

In [ ]:
def remove_tags(str):
  #remove HTML tags
  res = re.sub(r'<[^>]+>', '', str)

  #remove URLs
  res = re.sub(r'https?://\S+', '', res)

  #remove non-alphanumeric characters
  res = re.sub(r'[^a-zA-Z0-9' + r'\s]', '', res)

  #convert to lower case
  res = res.lower()
  return res

In [ ]:
df['review'] = df['review'].apply(remove_tags)

In [ ]:
nltk.download('stopwords')

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

df['review'] = df['review'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop_words)]))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
#nltk.download('wordnet')
#w_tokenizer = nltk.tokenize.WhitespaceTokenizer()
#lemmatizer = nltk.stem.WordNetLemmatizer()
#def lemmatize_text(text):
#    st = ""
#    for w in w_tokenizer.tokenize(text):
#        st = st + lemmatizer.lemmatize(w) + " "
#    return st
#df['review'] = df.review.apply(lemmatize_text)

In [ ]:
# split data into training data and test data
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42, stratify=df['sentiment'])

# **Tokenization**

In [ ]:
df.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,1
1,wonderful little production filming technique ...,1
2,thought wonderful way spend time hot summer we...,1
3,basically theres family little boy jake thinks...,0
4,petter matteis love time money visually stunni...,1


In [ ]:
# Tokenize text data
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_data["review"])
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review"]), maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review"]), maxlen=200)

In [ ]:
Y_train = train_data["sentiment"]
Y_test = test_data["sentiment"]

# **Build model**

In [ ]:
# build the model
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=128, input_length=200))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation="sigmoid"))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
# compile the model
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# **Train model**

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Define the EarlyStopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train the model with early stopping
model.fit(X_train, Y_train, epochs=10, batch_size=64, validation_split=0.2, callbacks=[early_stopping])

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 205s 384ms/step - accuracy: 0.7663 - loss: 0.4728 - val_accuracy: 0.8764 - val_loss: 0.2975
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 188s 377ms/step - accuracy: 0.8894 - loss: 0.2746 - val_accuracy: 0.8827 - val_loss: 0.2878
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 203s 378ms/step - accuracy: 0.9135 - loss: 0.2276 - val_accuracy: 0.8777 - val_loss: 0.3041
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 202s 378ms/step - accuracy: 0.9229 - loss: 0.1977 - val_accuracy: 0.8759 - val_loss: 0.3214
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 202s 378ms/step - accuracy: 0.9370 - loss: 0.1689 - val_accuracy: 0.8690 - val_loss: 0.3491


# **Prediction**

In [ ]:
# Validate the model
loss, accuracy = model.evaluate(X_test,Y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 34s 104ms/step - accuracy: 0.8779 - loss: 0.2934
Test Loss: 0.2904418408870697
Test Accuracy: 0.882099986076355


In [ ]:
def predict_sentiment(review):
  # tokenize and pad the review
  sequence = tokenizer.texts_to_sequences([review])
  padded_sequence = pad_sequences(sequence, maxlen=200)
  prediction = model.predict(padded_sequence)
  sentiment = "positive" if prediction[0][0] > 0.5 else "negative"
  return sentiment

In [ ]:
# example usage
new_review = "This movie was not that good"
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 435ms/step
The sentiment of the review is: positive


In [ ]:
from sklearn.metrics import classification_report
predicted_probabilities = model.predict(X_test)
predicted = (predicted_probabilities > 0.5).astype(int)
print(classification_report(Y_test, predicted))

313/313 ━━━━━━━━━━━━━━━━━━━━ 33s 105ms/step
              precision    recall  f1-score   support

           0       0.90      0.86      0.88      5000
           1       0.87      0.90      0.88      5000

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000

